In [3]:
import pandas as pd;
import numpy as np;
import matplotlib.pyplot as plt;
import seaborn as sns;
from sklearn.model_selection import train_test_split;
from sklearn.linear_model import LinearRegression;
from sklearn.metrics import mean_squared_error, r2_score;
from xgboost import XGBRegressor;
from sklearn.preprocessing import LabelEncoder;
from sklearn.ensemble import RandomForestRegressor;
from sklearn.metrics import mean_absolute_error;
import warnings
warnings.filterwarnings("ignore")
from lightgbm import LGBMRegressor;


In [4]:
# load the 2024 dataset only
dataset_files = [
    ('2024', '../classes/master_dataset_partie2_2024_stint.csv'),
]

frames = []
for year, path in dataset_files:
    try:
        current = pd.read_csv(path)
        if 'Year' not in current.columns:
            current['Year'] = int(year)
        frames.append(current)
        print(f'Loaded {path} -> {len(current)} rows')
    except FileNotFoundError:
        print(f'Skipped missing file: {path}')

if not frames:
    raise FileNotFoundError('Aucun fichier dataset 2024 introuvable')

df = pd.concat(frames, ignore_index=True)
print(df.info())
#check for missing values
print(df.isnull().sum())
print(df.describe())
print(df.dtypes)

Loaded ../classes/master_dataset_partie2_2024_stint.csv -> 18877 rows
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18877 entries, 0 to 18876
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Year                     18877 non-null  int64  
 1   Event                    18877 non-null  object 
 2   Driver                   18877 non-null  int64  
 3   Compound                 18877 non-null  object 
 4   CompoundEncoded          18877 non-null  int64  
 5   TyreLife                 18877 non-null  float64
 6   TrackTemp                18877 non-null  float64
 7   FuelLoad                 18877 non-null  float64
 8   Abrasivity               18877 non-null  float64
 9   LateralEnergy            18877 non-null  float64
 10  DeltaToBest              18877 non-null  float64
 11  LapNumber                18877 non-null  float64
 12  Stint                    18877 non-null  float64
 13  Correc

In [5]:
def compute_delta_correctly(df):
    required = {'Driver', 'Event', 'Stint', 'LapNumber', 'CorrectedLapTime_Global'}
    if not required.issubset(df.columns):
        return df

    df = df.sort_values(['Driver', 'Event', 'Stint', 'LapNumber']).copy()

    # Min progressif: seulement les tours passes
    df['BestCorrectedByStint'] = (
        df.groupby(['Driver', 'Event', 'Stint'])['CorrectedLapTime_Global']
        .transform(lambda x: x.expanding().min())
    )

    df['DeltaToBest'] = df['CorrectedLapTime_Global'] - df['BestCorrectedByStint']
    return df


def preprocess_data(df):
    df = df.copy()

    # Recompute DeltaToBest using progressive best-per-stint logic.
    df = compute_delta_correctly(df)

    # Support both possible naming conventions.
    delta_col = 'deltaToBest' if 'deltaToBest' in df.columns else 'DeltaToBest'

    # Exact compound per GP: Hard / Medium / Soft -> C1 to C5 depending on the circuit.
    F1_COMPOUNDS_2024 = {
        'Bahrain': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
        'Saudi Arabia': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
        'Australia': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Japan': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
        'China': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
        'Miami': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
        'Emilia-Romagna': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Monaco': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Canada': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Spain': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
        'Austria': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Great Britain': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
        'Hungary': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Belgium': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
        'Netherlands': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
        'Italy': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Azerbaijan': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Singapore': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'USA (Austin)': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
        'Mexico': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Brazil': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Las Vegas': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
        'Qatar': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
        'Abu Dhabi': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    }
    EVENT_TO_F1_KEY = {
        'Bahrain Grand Prix': 'Bahrain',
        'Saudi Arabian Grand Prix': 'Saudi Arabia',
        'Australian Grand Prix': 'Australia',
        'Japanese Grand Prix': 'Japan',
        'Chinese Grand Prix': 'China',
        'Miami Grand Prix': 'Miami',
        'Emilia Romagna Grand Prix': 'Emilia-Romagna',
        'Monaco Grand Prix': 'Monaco',
        'Canadian Grand Prix': 'Canada',
        'Spanish Grand Prix': 'Spain',
        'Austrian Grand Prix': 'Austria',
        'British Grand Prix': 'Great Britain',
        'Hungarian Grand Prix': 'Hungary',
        'Belgian Grand Prix': 'Belgium',
        'Dutch Grand Prix': 'Netherlands',
        'Italian Grand Prix': 'Italy',
        'Azerbaijan Grand Prix': 'Azerbaijan',
        'Singapore Grand Prix': 'Singapore',
        'United States Grand Prix': 'USA (Austin)',
        'Mexico City Grand Prix': 'Mexico',
        'Sao Paulo Grand Prix': 'Brazil',
        'Las Vegas Grand Prix': 'Las Vegas',
        'Qatar Grand Prix': 'Qatar',
        'Abu Dhabi Grand Prix': 'Abu Dhabi',
    }
    COMPUND_ALIASES = {
        'H': 'Hard',
        'M': 'Medium',
        'S': 'Soft',
        'HARD': 'Hard',
        'MEDIUM': 'Medium',
        'SOFT': 'Soft',
        'C1': 'C1',
        'C2': 'C2',
        'C3': 'C3',
        'C4': 'C4',
        'C5': 'C5',
    }

    def normalize_event_label(value):
        if pd.isna(value):
            return ''
        return (
            str(value).strip()
            .replace('São Paulo Grand Prix', 'Sao Paulo Grand Prix')
            .replace('Emilia-Romagna Grand Prix', 'Emilia Romagna Grand Prix')
        )

    def normalize_compound_label(value):
        if pd.isna(value):
            return np.nan
        text = str(value).strip()
        return COMPUND_ALIASES.get(text.upper(), text)

    # Encode season chronology from Bahrain 2023 to Abu Dhabi 2024.
    RACE_ORDER_2023_2024 = {
        (2023, 'Bahrain Grand Prix'): 1,
        (2023, 'Saudi Arabian Grand Prix'): 2,
        (2023, 'Australian Grand Prix'): 3,
        (2023, 'Azerbaijan Grand Prix'): 4,
        (2023, 'Miami Grand Prix'): 5,
        (2023, 'Monaco Grand Prix'): 6,
        (2023, 'Spanish Grand Prix'): 7,
        (2023, 'Canadian Grand Prix'): 8,
        (2023, 'Austrian Grand Prix'): 9,
        (2023, 'British Grand Prix'): 10,
        (2023, 'Hungarian Grand Prix'): 11,
        (2023, 'Belgian Grand Prix'): 12,
        (2023, 'Dutch Grand Prix'): 13,
        (2023, 'Italian Grand Prix'): 14,
        (2023, 'Singapore Grand Prix'): 15,
        (2023, 'Japanese Grand Prix'): 16,
        (2023, 'Qatar Grand Prix'): 17,
        (2023, 'United States Grand Prix'): 18,
        (2023, 'Mexico City Grand Prix'): 19,
        (2023, 'Sao Paulo Grand Prix'): 20,
        (2023, 'Las Vegas Grand Prix'): 21,
        (2023, 'Abu Dhabi Grand Prix'): 22,
        (2024, 'Bahrain Grand Prix'): 23,
        (2024, 'Saudi Arabian Grand Prix'): 24,
        (2024, 'Australian Grand Prix'): 25,
        (2024, 'Japanese Grand Prix'): 26,
        (2024, 'Chinese Grand Prix'): 27,
        (2024, 'Miami Grand Prix'): 28,
        (2024, 'Emilia Romagna Grand Prix'): 29,
        (2024, 'Monaco Grand Prix'): 30,
        (2024, 'Canadian Grand Prix'): 31,
        (2024, 'Spanish Grand Prix'): 32,
        (2024, 'Austrian Grand Prix'): 33,
        (2024, 'British Grand Prix'): 34,
        (2024, 'Hungarian Grand Prix'): 35,
        (2024, 'Belgian Grand Prix'): 36,
        (2024, 'Dutch Grand Prix'): 37,
        (2024, 'Italian Grand Prix'): 38,
        (2024, 'Azerbaijan Grand Prix'): 39,
        (2024, 'Singapore Grand Prix'): 40,
        (2024, 'United States Grand Prix'): 41,
        (2024, 'Mexico City Grand Prix'): 42,
        (2024, 'Sao Paulo Grand Prix'): 43,
        (2024, 'Las Vegas Grand Prix'): 44,
        (2024, 'Qatar Grand Prix'): 45,
        (2024, 'Abu Dhabi Grand Prix'): 46,
    }

    if {'Year', 'Event'}.issubset(df.columns):
        event_for_map = df['Event'].map(normalize_event_label)
        year_for_map = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
        race_keys = [
            (int(y), e) if pd.notna(y) else None
            for y, e in zip(year_for_map, event_for_map)
        ]
        df['RaceNumber'] = pd.Series(race_keys, index=df.index).map(RACE_ORDER_2023_2024)

    if {'Event', 'Compound'}.issubset(df.columns):
        event_for_compound = df['Event'].map(normalize_event_label)
        base_compound = df['Compound'].map(normalize_compound_label)
        exact_compound = [
            F1_COMPOUNDS_2024.get(EVENT_TO_F1_KEY.get(event, event), {}).get(base)
            for event, base in zip(event_for_compound, base_compound)
        ]
        df['CompoundExact'] = pd.Series(exact_compound, index=df.index)
        df['CompoundExact'] = df['CompoundExact'].fillna(base_compound)
        le_compound = LabelEncoder()
        df['CompoundEncoded'] = le_compound.fit_transform(df['CompoundExact'].astype(str))

    # Driver -> Team mapping for 2023 + 2024 (year-aware first, then fallback by driver)
    DRIVER_TEAM_BY_YEAR = {
        (2023, 'VER'): 'Red Bull', (2023, 'PER'): 'Red Bull',
        (2023, 'HAM'): 'Mercedes', (2023, 'RUS'): 'Mercedes',
        (2023, 'LEC'): 'Ferrari', (2023, 'SAI'): 'Ferrari',
        (2023, 'NOR'): 'McLaren', (2023, 'PIA'): 'McLaren',
        (2023, 'ALO'): 'Aston Martin', (2023, 'STR'): 'Aston Martin',
        (2023, 'OCO'): 'Alpine', (2023, 'GAS'): 'Alpine',
        (2023, 'BOT'): 'Sauber', (2023, 'ZHO'): 'Sauber',
        (2023, 'ALB'): 'Williams', (2023, 'SAR'): 'Williams',
        (2023, 'MAG'): 'Haas', (2023, 'HUL'): 'Haas',
        (2023, 'TSU'): 'RB', (2023, 'RIC'): 'RB',
        (2023, 'DEV'): 'RB', (2023, 'LAW'): 'RB',

        (2024, 'VER'): 'Red Bull', (2024, 'PER'): 'Red Bull',
        (2024, 'HAM'): 'Mercedes', (2024, 'RUS'): 'Mercedes',
        (2024, 'LEC'): 'Ferrari', (2024, 'SAI'): 'Ferrari',
        (2024, 'NOR'): 'McLaren', (2024, 'PIA'): 'McLaren',
        (2024, 'ALO'): 'Aston Martin', (2024, 'STR'): 'Aston Martin',
        (2024, 'OCO'): 'Alpine', (2024, 'GAS'): 'Alpine',
        (2024, 'BOT'): 'Sauber', (2024, 'ZHO'): 'Sauber',
        (2024, 'ALB'): 'Williams', (2024, 'SAR'): 'Williams',
        (2024, 'MAG'): 'Haas', (2024, 'HUL'): 'Haas',
        (2024, 'TSU'): 'RB', (2024, 'RIC'): 'RB',
    }
    DRIVER_TO_TEAM_FALLBACK = {
        'VER': 'Red Bull', 'PER': 'Red Bull',
        'HAM': 'Mercedes', 'RUS': 'Mercedes',
        'LEC': 'Ferrari', 'SAI': 'Ferrari',
        'NOR': 'McLaren', 'PIA': 'McLaren',
        'ALO': 'Aston Martin', 'STR': 'Aston Martin',
        'OCO': 'Alpine', 'GAS': 'Alpine',
        'BOT': 'Sauber', 'ZHO': 'Sauber',
        'ALB': 'Williams', 'SAR': 'Williams',
        'MAG': 'Haas', 'HUL': 'Haas',
        'TSU': 'RB', 'RIC': 'RB', 'DEV': 'RB', 'LAW': 'RB'
    }

    if {'Year', 'Driver'}.issubset(df.columns):
        year_for_team = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
        driver_for_team = df['Driver'].astype(str)
        team_keys = [
            (int(y), d) if pd.notna(y) else None
            for y, d in zip(year_for_team, driver_for_team)
        ]
        team_series = pd.Series(team_keys, index=df.index).map(DRIVER_TEAM_BY_YEAR)
        team_fallback = driver_for_team.map(DRIVER_TO_TEAM_FALLBACK)
        df['Team'] = team_series.fillna(team_fallback).fillna('Unknown')
    elif 'Driver' in df.columns:
        df['Team'] = df['Driver'].astype(str).map(DRIVER_TO_TEAM_FALLBACK).fillna('Unknown')

    if 'Team' in df.columns:
        le_team = LabelEncoder()
        df['TeamEncoded'] = le_team.fit_transform(df['Team'])

    if 'Event' in df.columns:
        le_event = LabelEncoder()
        df['EventEncoded'] = le_event.fit_transform(df['Event'])

    # Preserve season for later evaluation without using it as a feature.
    if 'Year' in df.columns:
        df['SeasonYear'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
    elif 'year' in df.columns:
        df['SeasonYear'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    # 1. Delta vitesse (variation intra-stint)
    if {'Driver', 'Stint', delta_col}.issubset(df.columns):
        df['delta_velocity'] = df.groupby(['Driver', 'Stint'])[delta_col].diff()

    # 2. Stress cumulatif
    df['lateral_stress_cumul'] = df['LateralEnergy'] * df['TyreLife']
    df['abrasive_stress_cumul'] = df['Abrasivity'] * df['TyreLife']

    # 3. Interaction thermique
    df['stress_x_temp'] = df['LateralEnergy'] * df['TrackTemp'] * df['TyreLife']

    # 4. Target
    if {'Driver', 'Stint', delta_col}.issubset(df.columns):
        df['delta_next_lap'] = df.groupby(['Driver', 'Stint'])[delta_col].shift(-1)

    # 5. Drop derniere ligne de chaque stint (target = NaN)
    if 'delta_next_lap' in df.columns:
        df = df.dropna(subset=['delta_next_lap'])

    # 6. Feature engineering that still needs Driver/Event before they are dropped
    if 'RaceNumber' in df.columns:
        df = df[df['RaceNumber'] != 6 ] 
        df=df[df["RaceNumber"] != 30 ]  # Monaco = course 6,30

    if {'CompoundEncoded', 'Abrasivity'}.issubset(df.columns):
        df['compound_x_abrasivity'] = df['CompoundEncoded'] * df['Abrasivity']
    if {'CompoundEncoded', 'LateralEnergy'}.issubset(df.columns):
        df['compound_x_lateral'] = df['CompoundEncoded'] * df['LateralEnergy']
    if {'CompoundEncoded', 'TyreLife'}.issubset(df.columns):
        df['compound_x_tyrelife'] = df['CompoundEncoded'] * df['TyreLife']

    if {'Driver', 'RaceNumber', 'Stint', 'LapNumber'}.issubset(df.columns):
        df = df.sort_values(['Driver', 'RaceNumber', 'Stint', 'LapNumber'])
        df['prev_stint_max_delta'] = (
            df.groupby(['Driver', 'RaceNumber'])
            .apply(lambda g: g.groupby('Stint')['DeltaToBest']
                   .max().shift(1).reindex(g.index, method='ffill'))
            .reset_index(level=[0, 1], drop=True)
        )

    if {'RaceNumber', 'Stint', 'TyreLife'}.issubset(df.columns):
        if 'Driver' in df.columns:
            df['stint_length'] = df.groupby(['Driver', 'RaceNumber', 'Stint'])['TyreLife'].transform('max')
        else:
            df['stint_length'] = df.groupby(['RaceNumber', 'Stint'])['TyreLife'].transform('max')
        df['tyre_life_pct'] = df['TyreLife'] / df['stint_length']

    # 6. Remove raw categorical columns after encoding
    df = df.drop(columns=['Event', 'Driver', 'Compound', 'CompoundExact', 'Year', 'year', 'Team', 'YearEncoded'], errors='ignore')

    return df

In [6]:
df = preprocess_data(df)
print(df.columns)

Index(['CompoundEncoded', 'TyreLife', 'TrackTemp', 'FuelLoad', 'Abrasivity',
       'LateralEnergy', 'DeltaToBest', 'LapNumber', 'Stint',
       'CorrectedLapTime_Global', 'BestCorrectedByStint', 'RaceNumber',
       'TeamEncoded', 'EventEncoded', 'SeasonYear', 'delta_velocity',
       'lateral_stress_cumul', 'abrasive_stress_cumul', 'stress_x_temp',
       'delta_next_lap', 'compound_x_abrasivity', 'compound_x_lateral',
       'compound_x_tyrelife', 'prev_stint_max_delta', 'stint_length',
       'tyre_life_pct'],
      dtype='object')


In [7]:
df = df.drop(columns=[
    'BestCorrectedByStint',     # leakage (utilise pour calculer DeltaToBest)
    'CorrectedLapTime_Global',  # leakage indirect
    'EventEncoded',             # redondant avec RaceNumber
], errors='ignore')

# Premier tour de chaque stint: pas de tour precedent
if 'delta_velocity' in df.columns:
    df['delta_velocity'] = df['delta_velocity'].fillna(0)

corr = df[['LapNumber', 'TyreLife']].corr().iloc[0, 1]
print(f"Correlation LapNumber/TyreLife : {corr:.3f}")

# Si > 0.85, supprimer LapNumber
if corr > 0.85:
    df = df.drop(columns=['LapNumber'], errors='ignore')

Correlation LapNumber/TyreLife : 0.536


In [21]:
import matplotlib.pyplot as plt

print(df['delta_next_lap'].describe())
#         mean   std    min     max
# Attendu: ~0.3  ~0.4   0.0    ~3.0

# Chercher les outliers
print(f"Valeurs > 3s : {(df['delta_next_lap'] > 3).sum()}")
print(f"Valeurs < 0  : {(df['delta_next_lap'] < 0).sum()}")

# Clipper les outliers extrêmes
df = df[df['delta_next_lap'].between(0, 3)]

count    17864.000000
mean         0.608277
std          0.608073
min          0.000000
25%          0.085000
50%          0.468000
75%          0.946648
max         11.998000
Name: delta_next_lap, dtype: float64
Valeurs > 3s : 65
Valeurs < 0  : 0


In [22]:
# Vérifier qu'il n'y a pas de valeurs aberrantes extrêmes
# qui pourraient ralentir l'apprentissage

print(df[['TyreLife', 'TrackTemp', 'FuelLoad', 
          'lateral_stress_cumul', 'stress_x_temp']].describe())

# stress_x_temp = lateral × TrackTemp × TyreLife
# peut atteindre 5 × 50 × 40 = 10000
# → pas un problème pour XGBoost mais vérifie que c'est cohérent



           TyreLife     TrackTemp      FuelLoad  lateral_stress_cumul  \
count  17799.000000  17799.000000  17799.000000          17799.000000   
mean      14.597112     35.967672     29.357380             48.597129   
std        9.312367      8.463936     17.737149             34.356293   
min        1.000000     16.700000      0.000000              1.800000   
25%        7.000000     30.700000     14.000000             22.400000   
50%       13.000000     37.700000     29.000000             41.000000   
75%       20.000000     42.800000     44.000000             65.600000   
max       57.000000     51.700000     70.000000            250.800000   

       stress_x_temp  
count   17799.000000  
mean     1716.248798  
std      1194.007322  
min        32.940000  
25%       771.045000  
50%      1452.800000  
75%      2412.560000  
max      7975.440000  


In [ ]:
from preprocess import preprocess

In [8]:
# Bloc déplacé dans preprocess_data pour eviter la dependance a Driver apres suppression
print(df.columns)

Index(['CompoundEncoded', 'TyreLife', 'TrackTemp', 'FuelLoad', 'Abrasivity',
       'LateralEnergy', 'DeltaToBest', 'LapNumber', 'Stint', 'RaceNumber',
       'TeamEncoded', 'SeasonYear', 'delta_velocity', 'lateral_stress_cumul',
       'abrasive_stress_cumul', 'stress_x_temp', 'delta_next_lap',
       'compound_x_abrasivity', 'compound_x_lateral', 'compound_x_tyrelife',
       'prev_stint_max_delta', 'stint_length', 'tyre_life_pct'],
      dtype='object')


In [23]:
import numpy as np

# Liste de toutes les courses 2024
all_races = np.array(sorted(df['RaceNumber'].unique()))

# Split fixe par course (pas par tour)
TEST_RACES = [25, 38, 23]  # fixer en dur
test_races = np.array(TEST_RACES)
train_races = np.array([r for r in all_races if r not in test_races])

print(f"Courses test  : {sorted(test_races)}")
print(f"Courses train : {sorted(train_races)}")

train = df[df['RaceNumber'].isin(train_races)]
test  = df[df['RaceNumber'].isin(test_races)]

X_train = train.drop(columns=['delta_next_lap', 'SeasonYear'], errors='ignore')
y_train = train['delta_next_lap']

X_test = test.drop(columns=['delta_next_lap', 'SeasonYear'], errors='ignore')
y_test = test['delta_next_lap']

print(f"Train : {len(train)} lignes")
print(f"Test  : {len(test)} lignes")

Courses test  : [np.int64(23), np.int64(25), np.int64(38)]
Courses train : [np.int64(24), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(39), np.int64(40), np.int64(42), np.int64(44), np.int64(45), np.int64(46)]
Train : 15139 lignes
Test  : 2660 lignes


In [24]:
df.columns

Index(['CompoundEncoded', 'TyreLife', 'TrackTemp', 'FuelLoad', 'Abrasivity',
       'LateralEnergy', 'DeltaToBest', 'LapNumber', 'Stint', 'RaceNumber',
       'TeamEncoded', 'SeasonYear', 'delta_velocity', 'lateral_stress_cumul',
       'abrasive_stress_cumul', 'stress_x_temp', 'delta_next_lap',
       'compound_x_abrasivity', 'compound_x_lateral', 'compound_x_tyrelife',
       'prev_stint_max_delta', 'stint_length', 'tyre_life_pct'],
      dtype='object')

In [20]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Mesure de la variance sur plusieurs splits course-par-course
# On réentraîne le même modèle à chaque seed pour voir la stabilité réelle

def train_and_evaluate_split(seed):
    rng = np.random.default_rng(seed)
    test_races = rng.choice(all_races, size=3, replace=False)
    train_races = np.array([r for r in all_races if r not in test_races])

    train = df[df['RaceNumber'].isin(train_races)]
    test = df[df['RaceNumber'].isin(test_races)]

    X_train = train.drop(columns=['delta_next_lap', 'SeasonYear'], errors='ignore')
    y_train = train['delta_next_lap']
    X_test = test.drop(columns=['delta_next_lap', 'SeasonYear'], errors='ignore')
    y_test = test['delta_next_lap']

    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=5,
        gamma=0.1,
        random_state=42,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)

    return {
        'seed': seed,
        'test_races': sorted(test_races.tolist()),
        'mae': mae,
        'n_train': len(train),
        'n_test': len(test),
    }

mae_results = []
for seed in range(10):
    result = train_and_evaluate_split(seed)
    mae_results.append(result)
    print(f"seed={seed:02d} | test={result['test_races']} | MAE={result['mae']:.4f}s | train={result['n_train']} | test_rows={result['n_test']}")

mae_df = pd.DataFrame(mae_results)
print()
print(f"MAE moyenne  : {mae_df['mae'].mean():.4f}s")
print(f"MAE std      : {mae_df['mae'].std():.4f}s")
print(f"MAE min      : {mae_df['mae'].min():.4f}s")
print(f"MAE max      : {mae_df['mae'].max():.4f}s")
print()
print(mae_df.sort_values('mae').reset_index(drop=True))

seed=00 | test=[34, 36, 40] | MAE=0.2601s | train=15581 | test_rows=2218
seed=01 | test=[32, 34, 39] | MAE=0.2826s | train=15458 | test_rows=2341
seed=02 | test=[25, 28, 39] | MAE=0.2252s | train=15327 | test_rows=2472
seed=03 | test=[24, 26, 39] | MAE=0.2796s | train=15642 | test_rows=2157
seed=04 | test=[37, 44, 46] | MAE=0.2486s | train=14936 | test_rows=2863
seed=05 | test=[23, 36, 40] | MAE=0.2595s | train=15124 | test_rows=2675
seed=06 | test=[32, 34, 46] | MAE=0.2787s | train=15362 | test_rows=2437
seed=07 | test=[36, 38, 42] | MAE=0.2327s | train=15214 | test_rows=2585
seed=08 | test=[27, 29, 37] | MAE=0.2388s | train=14745 | test_rows=3054
seed=09 | test=[32, 42, 46] | MAE=0.2674s | train=14875 | test_rows=2924

MAE moyenne  : 0.2573s
MAE std      : 0.0204s
MAE min      : 0.2252s
MAE max      : 0.2826s

   seed    test_races       mae  n_train  n_test
0     2  [25, 28, 39]  0.225156    15327    2472
1     7  [36, 38, 42]  0.232735    15214    2585
2     8  [27, 29, 37]  0.2388

In [48]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    'XGBoost': XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=5,
        gamma=0.1,
        random_state=42,
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        min_child_samples=20,
        reg_lambda=1.0,
        num_leaves=63,
        verbose=-1,
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
}

results = {}
trained_models = {}
predictions = {}

for name, current_model in models.items():
    if name == 'XGBoost':
        current_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    elif name == 'LightGBM':
        current_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
    else:
        current_model.fit(X_train, y_train)
    current_pred = current_model.predict(X_test)
    trained_models[name] = current_model
    predictions[name] = current_pred
    results[name] = {
        'MAE': mean_absolute_error(y_test, current_pred),
        'R2': r2_score(y_test, current_pred),
    }
    print(f"{name} -> MAE : {results[name]['MAE']:.4f}s | R² : {results[name]['R2']:.4f}")

# Garder XGBoost comme modèle actif pour les cellules suivantes
model = trained_models['XGBoost']
y_pred = predictions['XGBoost']

XGBoost -> MAE : 0.2197s | R² : 0.6628
LightGBM -> MAE : 0.2179s | R² : 0.6688
RandomForest -> MAE : 0.2166s | R² : 0.6816


In [39]:
# 1. Quelles courses sont dans le test
# Calculer la dégradation moyenne par circuit
circuit_deg = df.groupby('RaceNumber')['delta_next_lap'].mean().sort_values()
print(circuit_deg)

# Spliter en 3 tiers : faible / moyen / fort
n = len(circuit_deg)
low_deg    = circuit_deg.index[:n//3].tolist()
medium_deg = circuit_deg.index[n//3:2*n//3].tolist()
high_deg   = circuit_deg.index[2*n//3:].tolist()

# Prendre 1 circuit de chaque tier
import random
random.seed(42)
TEST_RACES = [
    random.choice(low_deg),
    random.choice(medium_deg),
    random.choice(high_deg),
]

print(f"Test races : {TEST_RACES}")
print(f"Dégradation test : {circuit_deg[TEST_RACES]}")

RaceNumber
31    0.083449
45    0.187497
24    0.198117
39    0.298972
28    0.326749
25    0.359231
44    0.390657
38    0.418709
34    0.435628
36    0.481142
46    0.508928
40    0.518987
42    0.615718
27    0.726038
23    0.758700
37    0.801243
33    0.817136
29    0.841653
26    0.854102
35    0.916780
32    0.917006
Name: delta_next_lap, dtype: float64
Test races : [25, 38, 23]
Dégradation test : RaceNumber
25    0.359231
38    0.418709
23    0.758700
Name: delta_next_lap, dtype: float64


In [ ]:
# Souvent aussi efficace que tout le reste
# en ciblant les 2 paramètres qui comptent vraiment
from sklearn.ensemble import RandomForestRegressor

results_grid = []
best_model = None
best_mae = float('inf')
best_params = None

for max_depth in [8, 10, 12]:
    for min_samples_leaf in [5, 8, 12]:
        rf = RandomForestRegressor(
            n_estimators=500,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=15,
            max_features=0.7,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred_rf = rf.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred_rf)

        results_grid.append({
            'max_depth': max_depth,
            'min_samples_leaf': min_samples_leaf,
            'MAE': mae
        })

        if mae < best_mae:
            best_mae = mae
            best_model = rf
            best_params = {
                'max_depth': max_depth,
                'min_samples_leaf': min_samples_leaf,
                'min_samples_split': 15,
                'max_features': 0.7,
                'n_estimators': 500,
                'bootstrap': True
            }

results_df = pd.DataFrame(results_grid).sort_values('MAE').reset_index(drop=True)
print(results_df)
print()
print(f"Best grid model -> MAE: {best_mae:.4f}s | params: {best_params}")


   max_depth  min_samples_leaf       MAE
0         10                12  0.211023
1          8                 5  0.211192
2         10                 8  0.211631
3          8                12  0.211689
4          8                 8  0.211712
5         12                12  0.212253
6         10                 5  0.212395
7         12                 8  0.213340
8         12                 5  0.213727

Best grid model -> MAE: 0.2110s | params: {'max_depth': 10, 'min_samples_leaf': 12, 'min_samples_split': 15, 'max_features': 0.7, 'n_estimators': 500, 'bootstrap': True}


In [ ]:
def plot_feature_importances_from_best_model(best_model, feature_names=None, top_n=20, figsize=(10, 6)):
    """
    Affiche et retourne les feature importances du meilleur modèle.

    Args:
        best_model: modèle entraîné (ex: RandomForestRegressor) avec attribut feature_importances_.
        feature_names: liste des noms de features. Si None, tente X_train.columns.
        top_n: nombre de variables les plus importantes à afficher.
        figsize: taille de la figure matplotlib.

    Returns:
        pd.DataFrame trié par importance décroissante.
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    if not hasattr(best_model, "feature_importances_"):
        raise AttributeError("Le modèle fourni ne possède pas feature_importances_.")

    if feature_names is None:
        if 'X_train' in globals() and hasattr(X_train, 'columns'):
            feature_names = list(X_train.columns)
        else:
            feature_names = [f"feature_{i}" for i in range(len(best_model.feature_importances_))]

    importances = np.asarray(best_model.feature_importances_)

    if len(feature_names) != len(importances):
        raise ValueError(
            f"Mismatch: {len(feature_names)} noms de features pour {len(importances)} importances."
        )

    fi_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    top_n = min(top_n, len(fi_df))
    top_df = fi_df.head(top_n).iloc[::-1]

    plt.figure(figsize=figsize)
    plt.barh(top_df["feature"], top_df["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Features")
    plt.title(f"Top {top_n} Feature Importances (Best Model)")
    plt.tight_layout()
    plt.show()

    return fi_df


# Exemple d'utilisation avec le meilleur modèle du grid search
feature_importances_df = plot_feature_importances_from_best_model(best_model, feature_names=X_train.columns, top_n=20)
feature_importances_df.head(20)

In [50]:
# Sauvegarder le meilleur modèle trouvé dans la cellule de grid search
import pickle

if 'best_model' not in globals():
    raise NameError('best_model is not defined. Exécute d’abord la cellule de grid search.')

with open('models/pace_predictor_rf_final_partie2.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print('✅ Meilleur modèle RF sauvegardé dans models/pace_predictor_rf_final_partie2.pkl')

✅ Meilleur modèle RF sauvegardé dans models/pace_predictor_rf_final_partie2.pkl


In [32]:
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint
from sklearn.metrics import mean_absolute_error, r2_score

param_dist = {
    'n_estimators':      randint(300, 800),
    'max_depth':         randint(6, 12),
    'min_samples_split': randint(5, 20),
    'min_samples_leaf':  randint(3, 15),
    'max_features':      [0.5, 0.7, 'sqrt', 'log2'],
}

search = HalvingRandomSearchCV(
    RandomForestRegressor(bootstrap=True, random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    factor=3,
    min_resources=500,
    random_state=42,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

mae = mean_absolute_error(y_test, search.best_estimator_.predict(X_test))
r2  = r2_score(y_test, search.best_estimator_.predict(X_test))

print(f"Best params : {search.best_params_}")
print(f"MAE test    : {mae:.4f}s")
print(f"R²  test    : {r2:.4f}")
print(f"Gain        : {0.250 - mae:+.4f}s")

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 500
max_resources_: 13290
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 26
n_resources: 500
Fitting 5 folds for each of 26 candidates, totalling 130 fits
----------
iter: 1
n_candidates: 9
n_resources: 1500
Fitting 5 folds for each of 9 candidates, totalling 45 fits
----------
iter: 2
n_candidates: 3
n_resources: 4500
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best params : {'max_depth': 11, 'max_features': 0.7, 'min_samples_leaf': 5, 'min_samples_split': 16, 'n_estimators': 354}
MAE test    : 0.2519s
R²  test    : 0.5944
Gain        : -0.0019s


In [24]:
y_pred_optuna = rf_optuna.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_optuna)
r2  = r2_score(y_test, y_pred_optuna)

print(f"MAE test : {mae:.4f}s")
print(f"R²  test : {r2:.4f}")
print(f"Gain     : {0.250 - mae:+.4f}s")

MAE test : 0.2943s
R²  test : 0.5201
Gain     : -0.0443s


In [20]:
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import GradientBoostingRegressor

stacking = StackingRegressor(
    estimators=[
        ('rf',  RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42)),
        ('et',  ExtraTreesRegressor(n_estimators=500, n_jobs=-1, random_state=42)),
        ('hgb', HistGradientBoostingRegressor(max_iter=500, random_state=42)),
        ('lgbm',LGBMRegressor(n_estimators=1000, n_jobs=-1, random_state=42, verbose=-1)),
    ],
    # Méta-modèle = petit RF au lieu de Ridge
    final_estimator=RandomForestRegressor(
        n_estimators=100, 
        random_state=42,
        n_jobs=-1
    ),
    cv=5,
    n_jobs=-1
)
stacking.fit(X_train, y_train)
y_pred_stack = stacking.predict(X_test)
print(f"Stacking MAE : {mean_absolute_error(y_test, y_pred_stack):.4f}s")
print(f"Stacking R²  : {r2_score(y_test, y_pred_stack):.4f}")

Stacking MAE : 0.2658s
Stacking R²  : 0.5648


In [ ]:
# 1. Résidus
y_pred = model.predict(X_test)
residuals = y_test - y_pred

print(pd.DataFrame({
    'y_test': y_test,
    'y_pred': y_pred,
    'residual': residuals
}).describe())

# 2. Feature importance en pourcentage (toutes les variables)
import pandas as pd
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importance_pct = (importance / importance.sum()) * 100
importance_df = importance_pct.rename('importance_pct').to_frame()
importance_df['importance_pct'] = importance_df['importance_pct'].round(2)
importance_df['importance_pct'] = importance_df['importance_pct'].map(lambda x: f"{x:.2f}%")
print(importance_df.head(10))  # top 10 features

In [ ]:
# Résultats sur la saison 2024
from sklearn.metrics import mean_absolute_error, r2_score

season_eval = test[['RaceNumber', 'delta_next_lap']].copy()
season_eval['SeasonYear'] = np.where(season_eval['RaceNumber'] >= 23, 2024, 2023)
season_eval['y_pred'] = y_pred

season_results = season_eval.groupby('SeasonYear').apply(lambda group: pd.Series({
    'MAE': mean_absolute_error(group['delta_next_lap'], group['y_pred']),
    'R2': r2_score(group['delta_next_lap'], group['y_pred']),
    'n': len(group)
}))

print(season_results.sort_index())

In [ ]:
# Entraînement sur les 10 meilleures variables (d'après le modèle actuel)
from sklearn.metrics import mean_absolute_error, r2_score

# Recalcule des importances en numérique pour sélectionner proprement le top 10
importance_num = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

top10_features = importance_num.head(10).index.tolist()
print('Top 10 features sélectionnées :')
print(top10_features)

# Sous-ensembles train/test sur ces 10 variables
X_train_top10 = X_train[top10_features].copy()
X_test_top10 = X_test[top10_features].copy()

# Nouveau modèle entraîné uniquement sur le top 10
model_top10 = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    min_child_samples=20,
    reg_lambda=1.0,
    num_leaves=63,
    verbose=-1,
    )

model_top10.fit(X_train_top10, y_train)
y_pred_top10 = model_top10.predict(X_test_top10)

print(f"MAE (Top 10) : {mean_absolute_error(y_test, y_pred_top10):.4f}s")
print(f"R²  (Top 10) : {r2_score(y_test, y_pred_top10):.4f}")

In [ ]:
# 1. Vérifier TeamEncoded
print(df.groupby('TeamEncoded')['delta_next_lap'].mean().sort_values())
# Tu dois voir des différences claires entre teams

# 2. Vérifier la variance par team
print(df.groupby('TeamEncoded')['delta_next_lap'].std().sort_values())

# 3. Scatter plot résidus vs TyreLife
import matplotlib.pyplot as plt
plt.scatter(X_test['TyreLife'], residuals, alpha=0.3)
plt.axhline(0, color='red')
plt.xlabel('TyreLife')
plt.ylabel('Résidual')
plt.title('Résidus vs TyreLife')
plt.show()
# Si pattern visible → TyreLife mal exploité

In [ ]:
# Ton modèle est meilleur sur les vieux pneus
# C'est exactement ce dont tu as besoin pour la stratégie pit stop

# Vérifie ça
mask_late = X_test['TyreLife'] > 15
print(f"MAE TyreLife > 15 : {mean_absolute_error(y_test[mask_late], y_pred[mask_late]):.4f}s")
print(f"MAE TyreLife < 15 : {mean_absolute_error(y_test[~mask_late], y_pred[~mask_late]):.4f}s")

# Tu vas voir quelque chose comme :
# MAE TyreLife > 15 : 0.18s  ✅
# MAE TyreLife < 15 : 0.35s  ⚠️

In [ ]:
# Regarder la distribution par tranche de TyreLife
import numpy as np

bins = [0, 5, 10, 15, 20, 25, 30, 50]
labels = ['1-5', '6-10', '11-15', '16-20', '21-25', '26-30', '30+']

X_test_copy = X_test.copy()
X_test_copy['y_test'] = y_test.values
X_test_copy['y_pred'] = y_pred
X_test_copy['bin'] = pd.cut(X_test_copy['TyreLife'], bins=bins, labels=labels)

print(X_test_copy.groupby('bin').agg(
    MAE=('y_test', lambda x: mean_absolute_error(
        x, X_test_copy.loc[x.index, 'y_pred'])),
    mean_delta=('y_test', 'mean'),
    count=('y_test', 'count')
))

In [16]:
if 'analysis_df' in globals() and 'importance_summary' in globals():
    worst = analysis_df.iloc[0]
    print('=== Synthèse compacte ===')
    print(
        f"Worst circuit: {worst['RaceName']} | MAE={worst['MAE']:.4f}s | "
        f"mean residual={worst['MeanResidual']:.4f}s | "
        f"underestimation rate={worst['UnderestimationRate']:.2%}"
    )
    print()
    print('Top 5 circuits by MAE:')
    print(analysis_df[['RaceName', 'MAE', 'MeanResidual', 'UnderestimationRate']].head(5).to_string(index=False))
    print()
    print('Importance stability:')
    print(importance_summary.loc[['TrackTempImportance', 'LateralEnergyImportance']].to_string())
else:
    print('analysis_df or importance_summary is missing; rerun the LOCO analysis cell first.')

=== Synthèse compacte ===
Worst circuit: Japanese Grand Prix | MAE=0.3574s | mean residual=-0.0081s | underestimation rate=51.86%

Top 5 circuits by MAE:
            RaceName      MAE  MeanResidual  UnderestimationRate
 Japanese Grand Prix 0.357356     -0.008129             0.518625
Las Vegas Grand Prix 0.331498     -0.130996             0.283019
  Spanish Grand Prix 0.330233      0.110444             0.545028
  British Grand Prix 0.301328     -0.066304             0.311538
 Canadian Grand Prix 0.292647     -0.247450             0.085427

Importance stability:
                             mean       std       min       max        cv
TrackTempImportance      0.022327  0.002345  0.015106  0.025041  0.105023
LateralEnergyImportance  0.050537  0.009466  0.015789  0.065513  0.187313
